<a href="https://colab.research.google.com/github/BozNatanium/AI_NLP_labs/blob/main/fine_tuning_hw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Домашнее задание

**Датасет:** [`ag_news`](https://huggingface.co/datasets/fancyzhx/ag_news) — классификация новостей по 4-м категориям (World, Sports, Business, Sci/Tech)

**Техническое задание:**

1.  Загрузите датасет `ag_news`
2.  Выберите модель для дообучения (например, `distilbert-base-uncased` или `bert-base-uncased`), `num_labels=4`
3.  Токенизируйте данные (`max_length=128`)
4.  Настройте `TrainingArguments`:
    *   `learning_rate = 2e-5`
    *   `per_device_train_batch_size = 16`
    *   `num_train_epochs = 3`
    *   `eval_strategy = "epoch"`
    *   `save_strategy = "epoch"`
    *   `load_best_model_at_end = True`
    *   `metric_for_best_model = "accuracy"`
5.  Обучите модель с помощью `Trainer`. Для метрик используйте `evaluate.load("accuracy")`
6.  Выведите accuracy на тестовой выборке
7.  Сохраните модель в папку `./ag_news_model`
8.  Протестируйте модель на трех новых новостях (вписать вручную), используя `pipeline`. Выведите предсказанный класс и уверенность модели

In [ ]:
!pip install transformers datasets evaluate torch accelerate -qq


In [ ]:
from datasets import load_dataset

dataset = load_dataset("fancyzhx/ag_news")
dataset


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

In [ ]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=32)

tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets


Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 7600
    })
})

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
import evaluate
import numpy as np

accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./ag_news_model",
    learning_rate=2e-5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    num_train_epochs=1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
)

trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy
1,0.245338,0.220564,0.924605


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1875, training_loss=0.2869489501953125, metrics={'train_runtime': 354.6282, 'train_samples_per_second': 338.383, 'train_steps_per_second': 5.287, 'total_flos': 993540925440000.0, 'train_loss': 0.2869489501953125, 'epoch': 1.0})

In [ ]:
metrics = trainer.evaluate()
print(f"Test accuracy: {metrics['eval_accuracy']:.4f}")


Training Loss,Validation Loss,Epoch,Accuracy
0.245338,0.220564,1,0.924605


Test accuracy: 0.9246


In [ ]:
model.save_pretrained("./ag_news_model")
tokenizer.save_pretrained("./ag_news_model")
print("Model saved to ./ag_news_model")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./ag_news_model


In [ ]:
from transformers import pipeline

classifier = pipeline("text-classification", model="./ag_news_model", tokenizer="./ag_news_model")

news_samples = [
    "Scientists discover a new exoplanet in the habitable zone of a nearby star system",
    "Stock markets hit new record highs as tech companies report strong quarterly earnings",
    "The home team won the championship after a thrilling overtime victory",
]

label_names = ["World", "Sports", "Business", "Sci/Tech"]

for text in news_samples:
    result = classifier(text)[0]
    label_idx = int(result["label"].split("_")[-1])
    print(f"News: {text}")
    print(f"Predicted: {label_names[label_idx]} (confidence: {result['score']:.4f})")
    print()


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

News: Scientists discover a new exoplanet in the habitable zone of a nearby star system
Predicted: Sci/Tech (confidence: 0.9662)

News: Stock markets hit new record highs as tech companies report strong quarterly earnings
Predicted: Business (confidence: 0.5021)

News: The home team won the championship after a thrilling overtime victory
Predicted: Sports (confidence: 0.9831)

